# Modèle d'éligibilité aux produits d'épargne — Notebook d'entraînement (V2.0)

**Projet** : Direction Data Engineering, Data Mining & Delivery — Banque Populaire
**Portée de ce notebook** : modélisation *pure Python* (pandas / scikit-learn / imbalanced-learn /
XGBoost / LightGBM / CatBoost / Optuna / SHAP) du **modèle d'éligibilité** aux produits d'épargne
(`Avenir Mes Enfants`, `MaRetraite`, `Épargne Evolution`).

## Positionnement dans le pipeline global

Ce notebook se situe **en aval** de la chaîne Big Data (NiFi → Spark/MLlib nettoyage → Parquet sur
MinIO). Il consomme un extrait déjà nettoyé (`dataset_eligibilite_final`, cf. `GUIDE_MAITRE`) sous
forme de `DataFrame` pandas — soit chargé directement (échantillon ou dataset réduit tenant en
mémoire), soit matérialisé en amont via Spark (`.toPandas()` / export Parquet) avant d'entrer ici.
La partie *distribution / infrastructure* (Spark, NiFi, Airflow, Docker) reste hors périmètre :
ce notebook traite la question **"comment obtenir le meilleur modèle de classification binaire
possible sur ces données"**, indépendamment du moteur qui les a produites.

## Pourquoi une réécriture complète (vs. `pipeline_training_v1_5.ipynb`)

Le notebook existant (V1.5) a rempli son rôle exploratoire mais accumule les limites d'un notebook
de travail : résultats figés en dur après des crashs mémoire (section 9bis), un seul run de
`RandomizedSearchCV` à 8 itérations pour XGBoost/LightGBM, pas de comparaison structurée
d'imputation, pas de sélection de variables formelle, pas de SHAP, pas d'analyse d'erreurs. Ce
notebook reprend les décisions déjà validées et documentées dans le `GUIDE_MAITRE` et le V1.5
(cf. encadrés *"Repris du V1.5"* ci-dessous) et les intègre dans une architecture plus complète,
plus robuste et testée plus largement — sans dupliquer le code Spark/MLlib, qui reste la référence
pour l'encodage en production.

> **Repris du V1.5 (validé, ne pas remettre en cause sans raison)**
> - Cible binaire déséquilibrée (~4.2 % de positifs, ratio ≈ 1:23).
> - `CODE_VILLE` est une variable catégorielle à haute cardinalité (~860+ modalités) — jamais en
>   One-Hot direct.
> - `BPR` est un **code** numérique sans relation d'ordre (12 modalités) → catégorielle, pas
>   quantité continue.
> - La pondération brute inverse-fréquence sur-corrige (rappel correct mais précision ~8-9 %) →
>   une variante adoucie (racine carrée) ou un rééquilibrage par sur/sous-échantillonnage donne de
>   meilleurs compromis précision/rappel.
> - Le seuil de décision à 0.5 n'a aucune raison d'être optimal sur une cible aussi déséquilibrée.

## Principe de non-fuite (data leakage)

Toute transformation qui *apprend* quelque chose des données (imputation, encodage, scaling,
sélection de variables, rééquilibrage) est encapsulée dans un `imblearn.pipeline.Pipeline`,
**fit exclusivement sur le train**. La validation croisée se fait sur ce pipeline complet — jamais
sur des features déjà transformées sur l'ensemble du dataset. Le rééquilibrage (SMOTE et
variantes) n'intervient **jamais** sur les folds de validation ni sur le test final.

## Plan du notebook

1. Introduction *(ce document)*
2. Objectif métier
3. Chargement des données
4. Configuration
5. Analyse exploratoire (EDA)
6. Nettoyage des données
7. Analyse des features
8. Feature engineering
9. Gestion des valeurs manquantes
10. Gestion du déséquilibre
11. Sélection des variables
12. Construction des pipelines
13. Optimisation des hyperparamètres
14. Entraînement des modèles
15. Comparaison des modèles
16. Optimisation du seuil de décision
17. Interprétabilité
18. Analyse des erreurs
19. Sauvegarde du meilleur modèle
20. Conclusion

## 2. Objectif métier

**Question** : un client donné est-il éligible à au moins un produit d'épargne de la gamme
(`Avenir Mes Enfants`, `MaRetraite`, `Épargne Evolution`) ?

- **Cible** : `label_eligibilite` ∈ {0, 1} — 0 = non éligible, 1 = éligible.
- **Déséquilibre** : ~4.2 % de positifs sur le dataset complet (3M+ lignes) — un modèle qui prédit
  toujours 0 atteint déjà ~95.8 % d'accuracy sans détecter un seul client éligible.

### Métrique d'optimisation

L'**accuracy est explicitement secondaire**. Ce que la banque veut, c'est identifier le plus
possible de clients réellement éligibles (rappel) sans noyer les commerciaux sous des faux
positifs (précision) — l'équilibre des deux est le **F1-score de la classe positive**, complété par
la **PR-AUC** (plus informative que la ROC-AUC sur une cible aussi déséquilibrée — cf. Davis &
Goadrich, 2006) pour comparer les modèles indépendamment d'un seuil.

| Priorité | Métrique | Rôle |
|---|---|---|
| 1 | **F1-score (classe 1)** | Critère de sélection du modèle et du seuil |
| 2 | **PR-AUC** | Comparaison des modèles indépendamment du seuil |
| 3 | **Recall (classe 1)** | Ne pas manquer les clients éligibles |
| 3 | **Precision (classe 1)** | Ne pas saturer les commerciaux de faux positifs |
| — | Accuracy, ROC-AUC | Suivies mais jamais utilisées pour arbitrer entre deux modèles |

### Contrainte métier implicite

Un faux négatif (client éligible non détecté) coûte une opportunité commerciale manquée ; un faux
positif coûte un contact commercial pour rien. En l'absence d'une matrice de coût métier chiffrée
communiquée par la Direction, le F1-score (moyenne harmonique, poids égal aux deux erreurs) est
retenu comme proxy raisonnable — **section 16** documente comment le seuil de décision peut être
redéplacé vers plus de rappel ou plus de précision si cette matrice de coût est précisée.

## 4. Configuration

Toutes les constantes du notebook sont regroupées ici : chemins, graine aléatoire, noms de
colonnes issus du contrat de données déjà établi (Spark/MLlib, `GUIDE_MAITRE`), et paramètres
globaux d'exécution (nombre de folds, budget d'optimisation, etc.). Centraliser ces valeurs évite
la duplication silencieuse qui avait déjà causé un bug dans le V1.5 (listes de colonnes
dupliquées entre deux notebooks, cf. `COLS_CATEGORIELLES_BASSE_CARDINALITE`).

In [1]:
from __future__ import annotations

import json
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

LOCAL_MODE = False

MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "minioadmin123")

MINIO_BUCKET = "processed-data"
MINIO_DATASET_PATH = "dataset_eligibilite_final/"

if LOCAL_MODE:
    DATA_PATH = Path(
        os.environ.get(
            "ELIGIBILITE_DATA_PATH",
            "./data/dataset_eligibilite_final.parquet"
        )
    )
else:
    DATA_PATH = f"s3://{MINIO_BUCKET}/{MINIO_DATASET_PATH}"

ARTIFACTS_DIR = Path("./artifacts")
MODELS_DIR = ARTIFACTS_DIR / "models"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
FIGURES_DIR = ARTIFACTS_DIR / "figures"

for d in (MODELS_DIR, REPORTS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

COL_LABEL = "label_eligibilite"
COLS_A_EXCLURE = ["label_code", COL_LABEL]

COLS_CATEGORIELLES_BASSE_CARDINALITE = [
    "GENDER", "TAILLE_ENTREPRI", "pack_actuel", "pack_etat",
    "CUSTOMER_RATING", "MARITAL_STATUS", "BPR",
]

COL_HAUTE_CARDINALITE = "CODE_VILLE"

N_FOLDS_CANDIDATS = [5, 10]
N_FOLDS_DEFAUT = 5
SEUILS_A_TESTER = np.round(np.arange(0.20, 0.91, 0.02), 2)
N_TRIALS_OPTUNA = 40
TEST_SIZE = 0.15
VAL_SIZE = 0.15

print("Configuration chargée.")

Configuration chargée.


In [3]:
!pip install fsspec s3fs pyarrow

     |████████████████████████████████| 193 kB 423 kB/s eta 0:00:01
     |████████████████████████████████| 1.3 MB 2.0 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 1.8 MB/s eta 0:00:011
     |████████████████████████████████| 243 kB 2.7 MB/s eta 0:00:01
     |████████████████████████████████| 319 kB 2.8 MB/s eta 0:00:01
     |████████████████████████████████| 129 kB 1.7 MB/s eta 0:00:01
     |████████████████████████████████| 84 kB 2.3 MB/s  eta 0:00:01
     |████████████████████████████████| 13.3 MB 6.4 MB/s eta 0:00:01
     |████████████████████████████████| 213 kB 4.8 MB/s eta 0:00:01
ERROR: s3transfer 0.11.5 has requirement botocore<2.0a.0,>=1.37.4, but you'll have botocore 1.37.3 which is incompatible.
ERROR: boto3 1.37.38 has requirement botocore<1.38.0,>=1.37.38, but you'll have botocore 1.37.3 which is incompatible.
ERROR: s3fs 2024.10.0 has requirement fsspec==2024.10.0.*, but you'll have fsspec 2025.3.0 which is incompatible.
  Attempting uninstall: botocore

## 3. Chargement des données

Le dataset attendu est le Parquet nettoyé produit par la chaîne Spark/MLlib
(`dataset_eligibilite_final`, cf. `GUIDE_MAITRE`) : catégorielles encore brutes (non encodées),
valeurs aberrantes déjà bornées par IQR, mais valeurs manquantes **non traitées** (c'est l'objet
de la section 9). `charger_donnees` accepte Parquet ou CSV et échoue explicitement si le fichier
est introuvable plutôt que de laisser une `FileNotFoundError` brute plus loin dans le notebook.

In [4]:
def charger_donnees(path):
    if str(path).startswith("s3://"):
        return pd.read_parquet(path)

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix == ".csv":
        return pd.read_csv(path)

    return pd.read_parquet(path)


df_raw = charger_donnees(DATA_PATH)

print(
    f"Dataset chargé : "
    f"{df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes"
    .replace(",", " ")
)

df_raw.head()

PermissionError: Forbidden

## 5. Analyse exploratoire (EDA)

Objectif : comprendre la structure du dataset avant toute transformation -- forme, types,
qualité (nulls, doublons, constantes), distribution de la cible, distribution de chaque feature,
corrélations, outliers, et lecture métier de chaque variable. Chaque sous-section produit à la
fois un résultat chiffré et une visualisation, pour repérer un problème d'un coup d'œil sur un
dataset de plusieurs millions de lignes.

### 5.1 Forme, types et qualité générale

In [ ]:
def rapport_qualite(df: pd.DataFrame) -> pd.DataFrame:
    '''Un tableau de synthèse par colonne : type, % de nulls, cardinalité, constance.'''
    rapport = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_nulls": df.isna().sum(),
        "pct_nulls": (df.isna().mean() * 100).round(2),
        "n_uniques": df.nunique(dropna=True),
        "pct_uniques": (df.nunique(dropna=True) / len(df) * 100).round(3),
    })
    rapport["constante"] = rapport["n_uniques"] <= 1
    rapport["quasi_constante"] = (rapport["n_uniques"] <= 2) & (~rapport["constante"])
    return rapport.sort_values("pct_nulls", ascending=False)


print(f"Shape                 : {df_raw.shape}")
print(f"Doublons (lignes)     : {df_raw.duplicated().sum():,}".replace(",", " "))
print(f"Mémoire (approx.)     : {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} Mo")

rapport = rapport_qualite(df_raw)
rapport

In [ ]:
colonnes_constantes = rapport.index[rapport["constante"]].tolist()
colonnes_quasi_constantes = rapport.index[rapport["quasi_constante"]].tolist()
colonnes_forte_nullite = rapport.index[rapport["pct_nulls"] > 40].tolist()

print(f"Colonnes constantes        ({len(colonnes_constantes)}) : {colonnes_constantes}")
print(f"Colonnes quasi-constantes  ({len(colonnes_quasi_constantes)}) : {colonnes_quasi_constantes}")
print(f"Colonnes >40% de nulls     ({len(colonnes_forte_nullite)}) : {colonnes_forte_nullite}")

**Lecture** : les colonnes constantes n'apportent aucune information (variance nulle) et
seront supprimées en section 6. Les colonnes quasi-constantes (une modalité archi-dominante) sont
suspectes mais pas systématiquement supprimées -- une variable à 99 % de zéros peut rester
discriminante sur le 1 % restant si celui-ci est corrélé à la cible (à vérifier section 7, pas
juste sur son taux de nullité).

### 5.2 Distribution de la cible et déséquilibre

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")

compte_cible = df_raw[COL_LABEL].value_counts().sort_index()
pct_cible = (compte_cible / len(df_raw) * 100).round(2)
ratio_desequilibre = compte_cible.max() / compte_cible.min()

print("Distribution de la cible :")
for classe, effectif in compte_cible.items():
    print(f"  classe {classe} : {effectif:,} ({pct_cible[classe]}%)".replace(",", " "))
print(f"Ratio de déséquilibre (majoritaire / minoritaire) : {ratio_desequilibre:.1f}:1")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=compte_cible.index.astype(str), y=compte_cible.values, ax=axes[0], color="#2980b9")
axes[0].set_title("Effectifs par classe")
axes[0].set_xlabel(COL_LABEL)
axes[0].bar_label(axes[0].containers[0], fmt="{:,.0f}")

axes[1].pie(compte_cible.values, labels=[f"classe {c}\n({p}%)" for c, p in zip(compte_cible.index, pct_cible)],
            colors=["#2980b9", "#e74c3c"], autopct=None, startangle=90)
axes[1].set_title("Part relative des classes")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_distribution_cible.png", dpi=120)
plt.show()

**Lecture métier** : un déséquilibre de cet ordre confirme le choix de métrique (section 2) --
l'accuracy serait mécaniquement excellente pour un modèle inutile. Ce constat pilote directement
les sections 10 (rééquilibrage), 13 (métrique d'optimisation) et 16 (seuil de décision).

### 5.3 Cardinalité des variables catégorielles

In [ ]:
cols_categorielles = COLS_CATEGORIELLES_BASSE_CARDINALITE + [COL_HAUTE_CARDINALITE]
cols_categorielles = [c for c in cols_categorielles if c in df_raw.columns]

cardinalites = df_raw[cols_categorielles].nunique().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 0.4 * len(cardinalites) + 1))
sns.barplot(x=cardinalites.values, y=cardinalites.index, ax=ax, color="#8e44ad")
ax.set_xscale("log")
ax.set_xlabel("Nombre de modalités (échelle log)")
ax.set_title("Cardinalité des variables catégorielles")
plt.tight_layout()
plt.show()

cardinalites.to_frame("n_modalites")

**Lecture** : `CODE_VILLE` se détache très nettement des autres catégorielles -- confirme le
traitement séparé déjà retenu côté Spark (indexation seule, pas de One-Hot direct, cf. section 8).
Les autres variables (`GENDER`, `MARITAL_STATUS`, `BPR`, ...) ont une cardinalité modeste,
compatible avec du One-Hot classique.

### 5.4 Catégories rares

In [ ]:
SEUIL_RARE = 0.01  # une modalité représentant moins de 1% des lignes est jugée "rare"

for col in COLS_CATEGORIELLES_BASSE_CARDINALITE:
    if col not in df_raw.columns:
        continue
    freqs = df_raw[col].value_counts(normalize=True, dropna=False)
    rares = freqs[freqs < SEUIL_RARE]
    if len(rares) > 0:
        print(f"{col:20s} : {len(rares)} modalité(s) rare(s) (<{SEUIL_RARE:.0%}) -- "
              f"{rares.index.tolist()}")
    else:
        print(f"{col:20s} : aucune modalité rare")

**Décision** : les modalités rares identifiées ici seront regroupées dans une catégorie
`"AUTRE"` en section 8 (feature engineering) -- cela stabilise l'encodage (évite qu'une modalité
vue 3 fois dans le train et jamais dans le test fasse dérailler le One-Hot/Target Encoding) sans
perdre d'information discriminante, puisque ces modalités sont trop rares pour être apprises
individuellement de toute façon.

### 5.5 Distribution des variables numériques (histogrammes, KDE, boxplots)

In [ ]:
cols_numeriques = [
    c for c in df_raw.select_dtypes(include=[np.number]).columns
    if c not in COLS_A_EXCLURE and c not in cols_categorielles
]
print(f"{len(cols_numeriques)} variables numériques : {cols_numeriques}")

n_cols_grille = 3
n_lignes_grille = int(np.ceil(len(cols_numeriques) / n_cols_grille))
fig, axes = plt.subplots(n_lignes_grille, n_cols_grille, figsize=(5 * n_cols_grille, 3.5 * n_lignes_grille))
axes = np.atleast_1d(axes).ravel()

for ax, col in zip(axes, cols_numeriques):
    sns.histplot(df_raw[col].dropna(), kde=True, ax=ax, color="#16a085")
    ax.set_title(col, fontsize=10)
for ax in axes[len(cols_numeriques):]:
    ax.axis("off")

plt.suptitle("Histogrammes + KDE -- variables numériques", y=1.0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_histogrammes_numeriques.png", dpi=120)
plt.show()

In [ ]:
fig, axes = plt.subplots(n_lignes_grille, n_cols_grille, figsize=(5 * n_cols_grille, 3.5 * n_lignes_grille))
axes = np.atleast_1d(axes).ravel()

for ax, col in zip(axes, cols_numeriques):
    sns.boxplot(x=df_raw[col].dropna(), ax=ax, color="#f39c12")
    ax.set_title(col, fontsize=10)
for ax in axes[len(cols_numeriques):]:
    ax.axis("off")

plt.suptitle("Boxplots -- variables numériques (repérage des outliers)", y=1.0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_boxplots_numeriques.png", dpi=120)
plt.show()

### 5.6 Asymétrie (skewness) et aplatissement (kurtosis)

In [ ]:
from scipy.stats import skew, kurtosis

forme_distribution = pd.DataFrame({
    "skewness": df_raw[cols_numeriques].apply(lambda s: skew(s.dropna())),
    "kurtosis": df_raw[cols_numeriques].apply(lambda s: kurtosis(s.dropna())),
}).sort_values("skewness", key=abs, ascending=False)

forme_distribution["asymetrie_forte"] = forme_distribution["skewness"].abs() > 1
print(f"Variables fortement asymétriques (|skew|>1) : "
      f"{forme_distribution.index[forme_distribution['asymetrie_forte']].tolist()}")
forme_distribution

**Décision** : les variables avec `|skewness| > 1` sont candidates à une transformation
(log / Yeo-Johnson, section 8) -- utile en particulier pour les modèles linéaires
(`LogisticRegression`) ; les modèles à base d'arbres (RandomForest, XGBoost, ...) sont invariants
aux transformations monotones et n'en tirent pas de bénéfice direct, mais ces transformations
restent appliquées de façon homogène dans le pipeline pour ne pas maintenir deux chemins de
features différents selon le modèle.

### 5.7 Détection des outliers (méthode IQR)

In [ ]:
def compte_outliers_iqr(s: pd.Series, k: float = 1.5) -> int:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    bas, haut = q1 - k * iqr, q3 + k * iqr
    return int(((s < bas) | (s > haut)).sum())


outliers_iqr = pd.Series(
    {c: compte_outliers_iqr(df_raw[c].dropna()) for c in cols_numeriques}
).sort_values(ascending=False)
outliers_iqr = outliers_iqr.to_frame("n_outliers")
outliers_iqr["pct_outliers"] = (outliers_iqr["n_outliers"] / len(df_raw) * 100).round(2)
outliers_iqr

**Note** : le dataset source (`dataset_eligibilite_final`) a déjà subi un bornage IQR en
amont côté Spark (Partie 1 EDA, `OUTLIER_BOUNDS_PATH`) -- les outliers restants ici sont donc
attendus comme résiduels/faibles. S'ils s'avéraient nombreux, la bonne correction serait côté
pipeline Spark (source de vérité des bornes), pas un second bornage silencieux ici qui
diviserait la logique en deux endroits.

### 5.8 Matrice de corrélation et variables fortement corrélées

In [ ]:
matrice_corr = df_raw[cols_numeriques + [COL_LABEL]].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(0.5 * len(matrice_corr) + 3, 0.5 * len(matrice_corr) + 2))
sns.heatmap(matrice_corr, cmap="coolwarm", center=0, annot=len(matrice_corr) <= 20, fmt=".2f",
            square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.7})
ax.set_title("Matrice de corrélation (Pearson)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_heatmap_correlation.png", dpi=120)
plt.show()

In [ ]:
SEUIL_CORR_FORTE = 0.85

paires_correlees = (
    matrice_corr.where(np.triu(np.ones(matrice_corr.shape), k=1).astype(bool))
    .stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={"level_0": "variable_1", "level_1": "variable_2"})
)
paires_fortes = paires_correlees[paires_correlees["correlation"].abs() > SEUIL_CORR_FORTE]
paires_fortes = paires_fortes.sort_values("correlation", key=abs, ascending=False)

print(f"{len(paires_fortes)} paire(s) de variables corrélées à plus de {SEUIL_CORR_FORTE} :")
paires_fortes

**Décision** : les paires fortement corrélées (hors corrélation avec `label_eligibilite`
lui-même, qui est le signal recherché) sont redondantes -- une seule des deux sera généralement
conservée par la sélection de variables (section 11), par exemple via l'élimination redondante de
`RFECV`/importance SHAP plutôt qu'une suppression manuelle ici qui pourrait retirer la variable la
plus utile par erreur.

### 5.9 Lecture métier des variables clés

Un résumé synthétique du **sens métier** de chaque variable, issu de l'analyse du dictionnaire de
données réalisée en amont (rapport LaTeX de cartographie des 21 fichiers bruts, cf. mémoire projet) :

| Variable | Sens métier | Remarque |
|---|---|---|
| `GENDER` | Sexe du client | Basse cardinalité, OHE direct |
| `MARITAL_STATUS` | Situation familiale | Corrélée à l'appétence pour `Avenir Mes Enfants` |
| `TAILLE_ENTREPRI` | Taille de l'entreprise employeuse | Proxy de stabilité de revenu |
| `pack_actuel` / `pack_etat` | Offre bancaire actuelle et son statut | Signal fort d'engagement client |
| `CUSTOMER_RATING` | Score de notation interne du client | Souvent corrélé à la cible -- à surveiller pour fuite de données (le rating a-t-il été calculé *après* l'attribution du produit d'épargne ?) |
| `BPR` | Code point de vente / agence | Catégorielle malgré son type numérique |
| `CODE_VILLE` | Ville de résidence/domiciliation | Haute cardinalité, encodage dédié |

> ⚠️ **Point de vigilance fuite de données** : si `CUSTOMER_RATING` est recalculé après la
> souscription d'un produit d'épargne, il encoderait indirectement la cible. Ce point doit être
> confirmé avec l'équipe métier avant mise en production -- documenté ici, traité si besoin en
> section 6 (retrait de la variable) plutôt que découvert après coup sur une performance
> anormalement élevée en section 15.

## 6. Nettoyage des données

Sur la base du constat de la section 5 : suppression des doublons exacts, des colonnes
constantes/quasi-constantes (aucune valeur discriminante possible), et retrait des colonnes
techniques qui ne sont pas des features (identifiants, colonnes de fuite potentielle identifiées
en 5.9). Chaque suppression est **journalisée** -- indispensable pour expliquer, six mois plus
tard, pourquoi telle colonne a disparu du dataset final.

In [ ]:
journal_nettoyage = []

def loguer(etape: str, avant: int, apres: int, detail: str = ""):
    journal_nettoyage.append({"etape": etape, "lignes_avant": avant, "lignes_apres": apres,
                               "lignes_retirees": avant - apres, "detail": detail})


df_clean = df_raw.copy()

# 1. Doublons exacts
n_avant = len(df_clean)
df_clean = df_clean.drop_duplicates()
loguer("Suppression doublons", n_avant, len(df_clean))

# 2. Colonnes constantes (aucune information)
if colonnes_constantes:
    df_clean = df_clean.drop(columns=colonnes_constantes)
    print(f"Colonnes constantes retirées : {colonnes_constantes}")

# 3. Colonnes à fuite de données potentielle -- désactivé par défaut, à activer
# après confirmation métier (cf. section 5.9). Laissé explicite plutôt que
# supprimé silencieusement, pour que la décision reste visible et traçable.
COLONNES_FUITE_SUSPECTEE: list[str] = []  # ex. ["CUSTOMER_RATING"] si confirmé par le métier
if COLONNES_FUITE_SUSPECTEE:
    df_clean = df_clean.drop(columns=[c for c in COLONNES_FUITE_SUSPECTEE if c in df_clean.columns])
    print(f"Colonnes retirées (fuite suspectée, confirmée métier) : {COLONNES_FUITE_SUSPECTEE}")

# 4. Lignes sans cible -- inutilisables pour l'entraînement supervisé
n_avant = len(df_clean)
df_clean = df_clean.dropna(subset=[COL_LABEL])
loguer("Suppression lignes sans cible", n_avant, len(df_clean))

# 5. Cible castée en entier propre (0/1) -- évite les surprises de type (bool, float, str)
df_clean[COL_LABEL] = df_clean[COL_LABEL].astype(int)

print(f"\nShape avant nettoyage : {df_raw.shape}")
print(f"Shape après nettoyage : {df_clean.shape}")
pd.DataFrame(journal_nettoyage)

## 7. Analyse des features

Au-delà de la description univariée (section 5), on regarde ici la relation de chaque variable
avec **la cible** -- c'est ce signal, pas la variance seule, qui doit guider le feature
engineering (section 8) et la sélection de variables (section 11).

In [ ]:
from scipy.stats import pointbiserialr, chi2_contingency

# --- Variables numériques vs cible : corrélation point-bisériale ----------
correlations_cible = {}
for col in cols_numeriques:
    valeurs = df_clean[[col, COL_LABEL]].dropna()
    if valeurs[col].nunique() > 1:
        r, p = pointbiserialr(valeurs[COL_LABEL], valeurs[col])
        correlations_cible[col] = {"correlation_point_biseriale": r, "p_value": p}

correlations_cible = pd.DataFrame(correlations_cible).T.sort_values(
    "correlation_point_biseriale", key=abs, ascending=False
)
correlations_cible

In [ ]:
fig, ax = plt.subplots(figsize=(7, max(3, 0.35 * len(correlations_cible))))
sns.barplot(x=correlations_cible["correlation_point_biseriale"], y=correlations_cible.index,
            ax=ax, palette="coolwarm")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Corrélation point-bisériale avec la cible -- variables numériques")
plt.tight_layout()
plt.show()

In [ ]:
# --- Variables catégorielles vs cible : V de Cramér (test du Chi²) --------
def v_cramer(col: str, cible: str, df: pd.DataFrame) -> float:
    table = pd.crosstab(df[col], df[cible])
    chi2, _, _, _ = chi2_contingency(table)
    n = table.sum().sum()
    r, k = table.shape
    return float(np.sqrt((chi2 / n) / (min(r - 1, k - 1) or 1)))


association_categorielles = pd.Series(
    {c: v_cramer(c, COL_LABEL, df_clean) for c in cols_categorielles if c in df_clean.columns}
).sort_values(ascending=False).to_frame("v_cramer")
association_categorielles

**Lecture** : le V de Cramér (∈ [0, 1]) mesure l'intensité d'association entre une
catégorielle et la cible, indépendamment du nombre de modalités -- contrairement au Chi² brut, qui
augmente mécaniquement avec la cardinalité. Les variables en tête de ce classement (et de la
corrélation point-bisériale ci-dessus) sont celles dont on attend le plus grand pouvoir prédictif
: à combiner avec les résultats de sélection de variables (section 11), mais déjà un bon indicateur
pour prioriser le feature engineering (section 8) sur les variables qui comptent réellement.

## 8. Feature engineering

### 8.1 Variables dérivées

Créées à partir de la lecture métier (section 5.9) et des variables numériques disponibles :
ratios, interactions entre variables fortement associées à la cible, regroupement des modalités
rares. Toutes ces transformations sont des **fonctions déterministes des features brutes** (pas
d'apprentissage de paramètres) -- elles peuvent donc être calculées avant le split
train/val/test sans risque de fuite.

In [ ]:
def creer_features_derivees(df: pd.DataFrame) -> pd.DataFrame:
    '''Construit des variables dérivées : ratios et interactions entre variables numériques
    disponibles, et regroupement des modalités catégorielles rares en \"AUTRE\".
    Fonction pure (pas de .fit()) -- appliquée identiquement train/val/test/production.'''
    df = df.copy()

    # --- Ratios entre paires de variables numériques strictement positives ---
    # (générique : n'suppose pas de noms de colonnes précis au-delà de ce qui est présent)
    cols_num_positives = [c for c in cols_numeriques if (df[c].dropna() > 0).all()]
    for i, c1 in enumerate(cols_num_positives):
        for c2 in cols_num_positives[i + 1:]:
            # Seulement les paires les plus corrélées à la cible (top 6 features numériques,
            # cf. section 7) -- évite une explosion combinatoire de ratios peu informatifs.
            top_features = correlations_cible.index[:6].tolist()
            if c1 in top_features and c2 in top_features:
                nom = f"ratio_{c1}_sur_{c2}"
                df[nom] = df[c1] / df[c2].replace(0, np.nan)

    # --- Regroupement des modalités rares (section 5.4) en "AUTRE" ------------
    for col in COLS_CATEGORIELLES_BASSE_CARDINALITE:
        if col not in df.columns:
            continue
        freqs = df[col].value_counts(normalize=True)
        modalites_rares = freqs[freqs < SEUIL_RARE].index
        if len(modalites_rares) > 0:
            df[col] = df[col].where(~df[col].isin(modalites_rares), other="AUTRE")

    return df


df_feat = creer_features_derivees(df_clean)
nouvelles_colonnes = [c for c in df_feat.columns if c not in df_clean.columns]
print(f"Variables dérivées créées ({len(nouvelles_colonnes)}) : {nouvelles_colonnes}")
df_feat[nouvelles_colonnes].describe().T if nouvelles_colonnes else "Aucune variable dérivée créée."

### 8.2 Transformation des variables asymétriques

Les variables identifiées en 5.6 (`|skewness| > 1`) sont transformées via **Yeo-Johnson**
(généralisation du Box-Cox qui accepte les valeurs négatives ou nulles -- contrairement au
Box-Cox qui exige des valeurs strictement positives, plus contraignant sur ce dataset). La
transformation est **apprise sur le train uniquement** (elle a un paramètre λ ajusté aux données)
: elle est donc intégrée au `Pipeline` scikit-learn (section 12), pas appliquée ici sur
l'ensemble du dataset — cette cellule sert uniquement à *visualiser* l'effet attendu, pas à
transformer `df_feat` définitivement.

In [ ]:
from sklearn.preprocessing import PowerTransformer

cols_a_transformer = forme_distribution.index[forme_distribution["asymetrie_forte"]].tolist()
cols_a_transformer = [c for c in cols_a_transformer if c in df_feat.columns]

if cols_a_transformer:
    demo_pt = PowerTransformer(method="yeo-johnson", standardize=False)
    demo_transforme = demo_pt.fit_transform(df_feat[cols_a_transformer].fillna(df_feat[cols_a_transformer].median()))

    fig, axes = plt.subplots(len(cols_a_transformer), 2, figsize=(9, 3 * len(cols_a_transformer)))
    axes = np.atleast_2d(axes)
    for i, col in enumerate(cols_a_transformer):
        sns.histplot(df_feat[col].dropna(), kde=True, ax=axes[i, 0], color="#c0392b")
        axes[i, 0].set_title(f"{col} -- brut (skew={forme_distribution.loc[col, 'skewness']:.2f})")
        sns.histplot(demo_transforme[:, i], kde=True, ax=axes[i, 1], color="#27ae60")
        axes[i, 1].set_title(f"{col} -- Yeo-Johnson")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "05_yeojohnson_avant_apres.png", dpi=120)
    plt.show()
    print(f"Variables retenues pour Yeo-Johnson dans le pipeline final : {cols_a_transformer}")
else:
    print("Aucune variable suffisamment asymétrique -- transformation non nécessaire.")

### 8.3 Choix de l'encodage des variables catégorielles

Quatre stratégies sont comparées par validation croisée rapide (`LogisticRegression`, modèle
sensible à l'encodage, sur un sous-échantillon stratifié pour rester rapide) :

- **One-Hot Encoding** — pour les catégorielles basse cardinalité (référence, aucune hypothèse
  d'ordre ni de lien avec la cible).
- **Ordinal Encoding** — rapide, mais impose un ordre arbitraire ; sert de repère bas plutôt que
  de candidat sérieux ici (aucune de nos catégorielles n'est réellement ordinale).
- **Frequency Encoding** — remplace chaque modalité par sa fréquence ; utile pour
  `CODE_VILLE` (haute cardinalité) car il ne fait pas exploser la dimensionnalité et reste
  {train,val,test}-cohérent (fréquences apprises sur le train uniquement).
- **Target Encoding** (moyenne de la cible par modalité, lissée) — puissant sur la haute
  cardinalité mais **à risque de fuite** s'il n'est pas fait avec un lissage/CV interne — utilisé
  ici via `category_encoders.TargetEncoder`, qui applique un lissage bayésien vers la moyenne
  globale.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

try:
    from category_encoders import TargetEncoder, CountEncoder
    CATEGORY_ENCODERS_DISPONIBLE = True
except ImportError:
    CATEGORY_ENCODERS_DISPONIBLE = False
    print("category_encoders indisponible -- installer avec "
          "`pip install category_encoders --break-system-packages` pour comparer Target/Frequency Encoding.")

# Sous-échantillon stratifié pour un comparatif rapide (pas besoin des 3M+ lignes ici)
echantillon_encodage = df_feat.sample(
    n=min(150_000, len(df_feat)), random_state=RANDOM_STATE
)
X_ech = echantillon_encodage.drop(columns=[c for c in COLS_A_EXCLURE if c in echantillon_encodage.columns])
y_ech = echantillon_encodage[COL_LABEL]

cols_cat_ech = [c for c in cols_categorielles if c in X_ech.columns]
cols_num_ech = [c for c in cols_numeriques + nouvelles_colonnes if c in X_ech.columns]

resultats_encodage = {}
cv_encodage = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def evaluer_encodeur(nom, encodeur_cat):
    preprocesseur = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), cols_num_ech),
        ("cat", SkPipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", encodeur_cat),
        ]), cols_cat_ech),
    ])
    pipe = SkPipeline([("prep", preprocesseur), ("clf", LogisticRegression(max_iter=200, class_weight="balanced"))])
    scores = cross_val_score(pipe, X_ech, y_ech, cv=cv_encodage, scoring="f1", n_jobs=-1)
    resultats_encodage[nom] = scores.mean()
    print(f"{nom:22s} F1(classe 1) = {scores.mean():.4f} (+/- {scores.std():.4f})")

evaluer_encodeur("One-Hot Encoding", OneHotEncoder(handle_unknown="ignore"))
evaluer_encodeur("Ordinal Encoding", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
if CATEGORY_ENCODERS_DISPONIBLE:
    evaluer_encodeur("Frequency Encoding", CountEncoder(normalize=True, handle_unknown=0, handle_missing=0))
    evaluer_encodeur("Target Encoding", TargetEncoder(smoothing=0.3))

meilleur_encodage = max(resultats_encodage, key=resultats_encodage.get)
print(f"\nMeilleur encodage retenu : {meilleur_encodage} (F1={resultats_encodage[meilleur_encodage]:.4f})")

**Décision retenue pour le pipeline final (section 12)** : One-Hot Encoding pour les
catégorielles basse cardinalité (comportement stable, interprétable, pas d'hypothèse
supplémentaire), **Frequency Encoding pour `CODE_VILLE`** (haute cardinalité — évite l'explosion
dimensionnelle du One-Hot et le risque de fuite du Target Encoding sur une variable à ~860
modalités où certaines n'apparaissent que quelques fois). Le résultat empirique ci-dessus
confirme ou nuance ce choix par défaut avant de le figer dans le pipeline.

## 9. Gestion des valeurs manquantes

Jamais de `fillna()` unique et arbitraire. Cinq stratégies sont comparées par validation croisée
(même protocole que pour l'encodage), **sur les variables numériques qui ont effectivement des
valeurs manquantes** (section 5.1) : moyenne, médiane, plus fréquente, `KNNImputer`,
`IterativeImputer` (MICE). Le choix est fait **par variable** si nécessaire — une variable très
asymétrique se prête mal à une simple moyenne, quand une variable proche de la normale s'en
satisfait très bien.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401 -- active IterativeImputer
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.ensemble import RandomForestRegressor

cols_avec_nulls = rapport.index[(rapport["pct_nulls"] > 0) & (rapport.index.isin(cols_numeriques))].tolist()
print(f"Variables numériques avec valeurs manquantes : {cols_avec_nulls}")

strategies_imputation = {
    "Mean": SimpleImputer(strategy="mean"),
    "Median": SimpleImputer(strategy="median"),
    "Most Frequent": SimpleImputer(strategy="most_frequent"),
    "KNN (k=5)": KNNImputer(n_neighbors=5),
    "Iterative (MICE, RF léger)": IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=25, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1),
        max_iter=5, random_state=RANDOM_STATE,
    ),
}

resultats_imputation = {}
if cols_avec_nulls:
    for nom, imputer in strategies_imputation.items():
        preprocesseur = ColumnTransformer([
            ("num", imputer, cols_num_ech),
            ("cat", SkPipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("encode", OneHotEncoder(handle_unknown="ignore")),
            ]), cols_cat_ech),
        ])
        pipe = SkPipeline([("prep", preprocesseur), ("clf", LogisticRegression(max_iter=200, class_weight="balanced"))])
        scores = cross_val_score(pipe, X_ech, y_ech, cv=cv_encodage, scoring="f1", n_jobs=-1)
        resultats_imputation[nom] = scores.mean()
        print(f"{nom:28s} F1(classe 1) = {scores.mean():.4f} (+/- {scores.std():.4f})")

    meilleure_imputation = max(resultats_imputation, key=resultats_imputation.get)
    print(f"\nMeilleure stratégie d'imputation : {meilleure_imputation} (F1={resultats_imputation[meilleure_imputation]:.4f})")
else:
    print("Aucune valeur manquante détectée sur les variables numériques -- section informative, "
          "aucune imputation nécessaire pour ce run.")

**Règle retenue pour le pipeline final (section 12)** :
- Catégorielles : `most_frequent` (une valeur manquante sur une catégorielle basse cardinalité est
  raisonnablement remplacée par le mode — alternative : une modalité `"MANQUANT"` dédiée si le
  volume de nulls est important et potentiellement informatif, à tester si le taux de nulls
  dépasse 5 % sur une colonne donnée, cf. rapport section 5.1).
- Numériques : la stratégie gagnante ci-dessus, avec **repli automatique sur `median`** si
  `KNNImputer`/`IterativeImputer` s'avère trop coûteux à l'échelle de production (ils nécessitent
  de conserver tout ou partie du train en mémoire pour imputer de nouvelles lignes — à valider
  avec la contrainte de volumétrie de scoring, section 19).
- **Fit strictement sur le train** : tout imputer est un stage du `Pipeline` (section 12), jamais
  calculé sur le dataset complet avant le split — c'est le principal vecteur de fuite de données
  sur cette étape.

## Split Train / Validation / Test

Trois blocs stratifiés, strictement séparés, avant toute étape qui *apprend* quelque chose des
données :

- **Train** (70 %) : sert au rééquilibrage (section 10), à la sélection de variables (section 11)
  et à l'entraînement de tous les modèles (section 14).
- **Validation** (15 %) : sert à l'optimisation des hyperparamètres (section 13) et à
  l'optimisation du seuil de décision (section 16). Jamais utilisée pour entraîner.
- **Test** (15 %) : n'est touché qu'une seule fois, en section 15, pour l'évaluation finale
  honnête du modèle retenu. Si le test est réutilisé pour re-choisir un modèle après l'avoir vu,
  il perd sa valeur d'évaluation indépendante — règle strictement respectée dans ce notebook.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_feat.drop(columns=[c for c in COLS_A_EXCLURE if c in df_feat.columns])
y = df_feat[COL_LABEL].astype(int)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=VAL_SIZE / (1 - TEST_SIZE), stratify=y_train_full, random_state=RANDOM_STATE
)

for nom, (Xs, ys) in {"Train": (X_train, y_train), "Validation": (X_val, y_val), "Test": (X_test, y_test)}.items():
    print(f"{nom:12s} : {len(Xs):>10,} lignes -- positifs = {ys.mean():.3%}".replace(",", " "))

cols_num_final = [c for c in cols_numeriques + nouvelles_colonnes if c in X_train.columns]
cols_cat_basse_final = [c for c in COLS_CATEGORIELLES_BASSE_CARDINALITE if c in X_train.columns]
cols_cat_haute_final = [c for c in [COL_HAUTE_CARDINALITE] if c in X_train.columns]

## 10. Gestion du déséquilibre

Trois familles de stratégies sont comparées, **toujours à l'intérieur d'une validation croisée**
(le rééquilibrage n'est appliqué que sur les folds d'entraînement, jamais sur le fold de
validation — sinon la métrique de comparaison est optimiste et invalide) :

1. **Sous-échantillonnage de la classe majoritaire** — plusieurs tailles cibles.
2. **Sur-échantillonnage de la classe minoritaire** — plusieurs variantes de SMOTE et apparentés.
3. **Nettoyage de frontière** — combinaisons de sur-échantillonnage + nettoyage (Tomek Links, ENN).

Un modèle de référence rapide (`LogisticRegression`, faible coût) sert de juge commun à toutes les
combinaisons, sur les features déjà imputées/encodées (section 8/9) — l'objectif ici est de choisir
la **stratégie de rééquilibrage**, pas encore le meilleur modèle final (section 14).

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import (
    RandomUnderSampler, TomekLinks, EditedNearestNeighbours, NearMiss, ClusterCentroids,
)
from imblearn.over_sampling import (
    RandomOverSampler, SMOTE, BorderlineSMOTE, SVMSMOTE, KMeansSMOTE, ADASYN, SMOTENC,
)
from imblearn.combine import SMOTETomek, SMOTEENN

# Préprocesseur commun (imputation + encodage) réutilisé pour toutes les comparaisons de cette
# section -- construit une fois, jamais fitté ici (le fit se fait dans chaque cross_val_score,
# séparément par fold, à l'intérieur du Pipeline imblearn).
def construire_preprocesseur():
    return ColumnTransformer([
        ("num", SkPipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("power", PowerTransformer(method="yeo-johnson")),
        ]), cols_num_final),
        ("cat_basse", SkPipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), cols_cat_basse_final),
        ("cat_haute", SkPipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]), cols_cat_haute_final),
    ])


# Index des colonnes catégorielles APRÈS assemblage numérique+OHE -- nécessaire pour SMOTENC,
# qui doit savoir quelles colonnes du tableau final sont catégorielles.
def indices_categoriels_apres_ohe(preprocesseur_fit) -> list:
    noms = preprocesseur_fit.get_feature_names_out()
    return [i for i, n in enumerate(noms) if n.startswith("cat_haute__")]


strategies_desequilibre = {
    "Baseline (aucun rééquilibrage)": None,
    "RandomUnderSampler (200k)": RandomUnderSampler(sampling_strategy={0: 200_000}, random_state=RANDOM_STATE),
    "RandomUnderSampler (300k)": RandomUnderSampler(sampling_strategy={0: 300_000}, random_state=RANDOM_STATE),
    "RandomOverSampler": RandomOverSampler(random_state=RANDOM_STATE),
    "SMOTE": SMOTE(random_state=RANDOM_STATE, n_jobs=-1),
    "BorderlineSMOTE": BorderlineSMOTE(random_state=RANDOM_STATE, n_jobs=-1),
    "ADASYN": ADASYN(random_state=RANDOM_STATE, n_jobs=-1),
    "SMOTE + Tomek Links": SMOTETomek(random_state=RANDOM_STATE),
    "SMOTE + ENN": SMOTEENN(random_state=RANDOM_STATE),
    "Tomek Links seul": TomekLinks(n_jobs=-1),
    "Edited Nearest Neighbours": EditedNearestNeighbours(n_jobs=-1),
}
# SVMSMOTE / KMeansSMOTE / NearMiss / ClusterCentroids : coûteux sur un dataset de plusieurs
# millions de lignes (calculs de distances/clustering) -- ajoutés au comparatif seulement sur
# l'échantillon réduit ci-dessous, jamais sur le dataset complet en production.
strategies_desequilibre_echantillon_uniquement = {
    "SVMSMOTE": SVMSMOTE(random_state=RANDOM_STATE, n_jobs=-1),
    "KMeansSMOTE": KMeansSMOTE(random_state=RANDOM_STATE),
    "NearMiss (v2)": NearMiss(version=2, n_jobs=-1),
    "ClusterCentroids": ClusterCentroids(random_state=RANDOM_STATE),
}

In [ ]:
# Comparatif sur un échantillon stratifié du train (coût calcul raisonnable) -- cf. note
# ci-dessus sur les stratégies coûteuses. `RandomForestClassifier` léger comme juge, plus robuste
# que la LogisticRegression aux features non standardisées issues de l'échantillon.
from sklearn.ensemble import RandomForestClassifier as _RFJuge

X_train_ech = X_train.sample(n=min(200_000, len(X_train)), random_state=RANDOM_STATE)
y_train_ech = y_train.loc[X_train_ech.index]

resultats_desequilibre = {}
toutes_strategies = {**strategies_desequilibre, **strategies_desequilibre_echantillon_uniquement}
cv_desequilibre = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

for nom, sampler in toutes_strategies.items():
    stages = [("prep", construire_preprocesseur())]
    if sampler is not None:
        stages.append(("resample", sampler))
    stages.append(("clf", _RFJuge(n_estimators=150, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)))
    pipe = ImbPipeline(stages)
    try:
        scores = cross_val_score(pipe, X_train_ech, y_train_ech, cv=cv_desequilibre, scoring="f1", n_jobs=1)
        resultats_desequilibre[nom] = scores.mean()
        print(f"{nom:35s} F1(classe 1) = {scores.mean():.4f} (+/- {scores.std():.4f})")
    except Exception as exc:
        print(f"{nom:35s} ÉCHEC ({exc.__class__.__name__}: {exc})")

tableau_desequilibre = pd.Series(resultats_desequilibre).sort_values(ascending=False).to_frame("f1_classe1_cv")
tableau_desequilibre

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.35 * len(tableau_desequilibre) + 1))
sns.barplot(x=tableau_desequilibre["f1_classe1_cv"], y=tableau_desequilibre.index, ax=ax, color="#d35400")
ax.set_title("F1 (classe 1, CV) par stratégie de rééquilibrage")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_comparatif_reequilibrage.png", dpi=120)
plt.show()

MEILLEURE_STRATEGIE_DESEQUILIBRE = tableau_desequilibre.index[0]
print(f"\nStratégie de rééquilibrage retenue pour le pipeline final : {MEILLEURE_STRATEGIE_DESEQUILIBRE}")

**Décision** : la stratégie en tête du classement ci-dessus est retenue par défaut dans le
pipeline final (section 12/14). Elle reste **paramétrable** — `RandomForestClassifier` sert de
juge générique ici, mais certains modèles (notamment les boosting avec `scale_pos_weight` /
`class_weight`) peuvent tirer plus de bénéfice d'une pondération native que d'un
sur-échantillonnage explicite ; les deux approches sont donc comparées à nouveau, modèle par
modèle, en section 14.

## 11. Sélection des variables

Plusieurs méthodes, de nature différente (filtre statistique, wrapper, importance de modèle),
sont comparées pour identifier un socle de variables réellement utiles — objectif double : réduire
le bruit/la dimensionnalité (bénéfique en particulier pour `LogisticRegression`) et accélérer
l'optimisation d'hyperparamètres (section 13) sur un dataset de plusieurs millions de lignes.

In [ ]:
from sklearn.feature_selection import (
    VarianceThreshold, mutual_info_classif, chi2, f_classif, SelectKBest, RFECV,
)
from sklearn.inspection import permutation_importance

# Matrice transformée une fois (imputation + encodage), réutilisée par toutes les méthodes de
# cette section -- fit sur le train uniquement.
preprocesseur_selection = construire_preprocesseur()
X_train_transforme = preprocesseur_selection.fit_transform(X_train_ech, y_train_ech)
noms_features_transformees = preprocesseur_selection.get_feature_names_out()
X_train_transforme = pd.DataFrame(
    X_train_transforme.toarray() if hasattr(X_train_transforme, "toarray") else X_train_transforme,
    columns=noms_features_transformees,
)

resultats_selection = {}

# 1. Variance Threshold -- élimine les colonnes quasi constantes après encodage
vt = VarianceThreshold(threshold=0.001)
vt.fit(X_train_transforme)
resultats_selection["Variance Threshold"] = set(noms_features_transformees[vt.get_support()])

# 2. Information mutuelle
mi = mutual_info_classif(X_train_transforme, y_train_ech, random_state=RANDOM_STATE)
top_mi = pd.Series(mi, index=noms_features_transformees).sort_values(ascending=False)
resultats_selection["Mutual Information (top 20)"] = set(top_mi.index[:20])

# 3. ANOVA F-test
f_scores, _ = f_classif(X_train_transforme, y_train_ech)
top_anova = pd.Series(f_scores, index=noms_features_transformees).sort_values(ascending=False)
resultats_selection["ANOVA F-test (top 20)"] = set(top_anova.index[:20])

# 4. Chi² -- exige des valeurs non négatives : appliqué uniquement sur les colonnes OHE (0/1)
cols_non_negatives = [c for c in noms_features_transformees if X_train_transforme[c].min() >= 0]
if cols_non_negatives:
    chi2_scores, _ = chi2(X_train_transforme[cols_non_negatives], y_train_ech)
    top_chi2 = pd.Series(chi2_scores, index=cols_non_negatives).sort_values(ascending=False)
    resultats_selection["Chi² (top 20)"] = set(top_chi2.index[:20])

print("Tailles des sous-ensembles retenus par méthode :")
for nom, ens in resultats_selection.items():
    print(f"  {nom:32s} : {len(ens)} variables")

In [ ]:
# 5. Importance native LightGBM/XGBoost + Permutation Importance -- entraînés une fois sur
# l'échantillon transformé, réutilisés aussi comme référence pour SHAP (section 17).
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

lgbm_pour_selection = LGBMClassifier(
    n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced", verbosity=-1
).fit(X_train_transforme, y_train_ech)
importance_lgbm = pd.Series(
    lgbm_pour_selection.feature_importances_, index=noms_features_transformees
).sort_values(ascending=False)
resultats_selection["LightGBM Importance (top 20)"] = set(importance_lgbm.index[:20])

xgb_pour_selection = XGBClassifier(
    n_estimators=200, random_state=RANDOM_STATE, eval_metric="logloss",
    scale_pos_weight=(y_train_ech == 0).sum() / max((y_train_ech == 1).sum(), 1),
).fit(X_train_transforme, y_train_ech)
importance_xgb = pd.Series(
    xgb_pour_selection.feature_importances_, index=noms_features_transformees
).sort_values(ascending=False)
resultats_selection["XGBoost Importance (top 20)"] = set(importance_xgb.index[:20])

perm = permutation_importance(
    lgbm_pour_selection, X_train_transforme, y_train_ech,
    scoring="f1", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)
importance_permutation = pd.Series(perm.importances_mean, index=noms_features_transformees).sort_values(ascending=False)
resultats_selection["Permutation Importance (top 20)"] = set(importance_permutation.index[:20])

# --- Consensus : variables retenues par au moins la moitié des méthodes -----
from collections import Counter
votes = Counter()
for ens in resultats_selection.values():
    votes.update(ens)
n_methodes = len(resultats_selection)
VARIABLES_RETENUES = sorted([v for v, c in votes.items() if c >= n_methodes / 2])
print(f"\n{len(VARIABLES_RETENUES)} variables retenues par consensus (>= 50% des méthodes) "
      f"sur {len(noms_features_transformees)} au total :")
print(VARIABLES_RETENUES)

**RFECV (référence, coûteuse)** : `RFECV` élimine récursivement les variables les moins
importantes avec validation croisée intégrée pour choisir automatiquement le nombre optimal de
variables. Coûteux sur un dataset de plusieurs millions de lignes — exécuté ici **uniquement sur
l'échantillon réduit**, à titre de validation du consensus ci-dessus plutôt que comme méthode
principale.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

estimateur_leger = DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE, class_weight="balanced")
rfecv = RFECV(estimateur_leger, step=0.1, cv=3, scoring="f1", n_jobs=-1, min_features_to_select=10)
rfecv.fit(X_train_transforme, y_train_ech)

variables_rfecv = set(noms_features_transformees[rfecv.support_])
print(f"RFECV : {rfecv.n_features_} variables optimales retenues.")
recouvrement = len(variables_rfecv & set(VARIABLES_RETENUES)) / max(len(variables_rfecv), 1)
print(f"Recouvrement avec le consensus multi-méthodes : {recouvrement:.0%}")

**Décision retenue** : `VARIABLES_RETENUES` (consensus multi-méthodes) sert de base pour le
`SelectFromModel`/masque de colonnes intégré au pipeline final (section 12) — un fort
recouvrement avec `RFECV` renforce la confiance dans ce sous-ensemble. Les variables non
retenues restent documentées (pas supprimées du dataset brut) au cas où une évolution future du
modèle souhaiterait les réintégrer.

## 12. Construction des pipelines

Un seul pipeline `imblearn.pipeline.Pipeline` (pas `sklearn.pipeline.Pipeline` — celui-ci ne sait
pas enchaîner une étape de rééquilibrage, qui change le nombre de lignes) encapsule **toutes**
les étapes apprises :

`Imputation → Transformation (Yeo-Johnson) → Encodage → Sélection de variables → Rééquilibrage → Modèle`

Chaque étape n'est fit que sur les données qui lui sont présentées au moment du `.fit()` global —
sur un fold de validation croisée, cela veut dire : fit uniquement sur les folds d'entraînement de
CE fold, jamais sur le fold de test de CE fold. C'est cette discipline qui élimine la fuite de
données évoquée dans les contraintes du projet.

In [ ]:
from sklearn.feature_selection import SelectFromModel

def construire_pipeline(modele, sampler=None, avec_selection=True):
    '''Construit un pipeline complet et cohérent pour un modèle donné.

    - modele      : un estimateur scikit-learn-compatible (déjà paramétré).
    - sampler     : stratégie de rééquilibrage (section 10) -- None pour les modèles qui gèrent
                    le déséquilibre nativement (class_weight / scale_pos_weight).
    - avec_selection : si True, insère un SelectFromModel basé sur un LightGBM léger, cohérent
                    avec le consensus de sélection de variables (section 11).
    '''
    stages = [("prep", construire_preprocesseur())]

    if avec_selection:
        stages.append((
            "select",
            SelectFromModel(
                LGBMClassifier(n_estimators=100, random_state=RANDOM_STATE, verbosity=-1,
                                class_weight="balanced"),
                threshold="median",
            ),
        ))

    if sampler is not None:
        stages.append(("resample", sampler))

    stages.append(("clf", modele))
    return ImbPipeline(stages)


print("`construire_pipeline(modele, sampler, avec_selection)` prête à l'emploi pour la section 14.")

## 13. Optimisation des hyperparamètres

`GridSearchCV` est écarté (coût combinatoire prohibitif sur un dataset de plusieurs millions de
lignes). **Optuna** (recherche bayésienne, `TPESampler`) optimise directement le **F1-score de la
classe positive**, avec un budget de `N_TRIALS_OPTUNA` essais par modèle et un `pruner` qui arrête
tôt les essais manifestement mauvais — plus efficace qu'un `RandomizedSearchCV` à budget égal, en
particulier sur des espaces d'hyperparamètres à forte dimension (XGBoost, LightGBM, CatBoost).

L'optimisation se fait sur le **train** (validation croisée interne à Optuna, `StratifiedKFold` à
`N_FOLDS_DEFAUT` folds) — jamais sur `X_val`/`X_test`, qui restent réservés respectivement au choix
du seuil (section 16) et à l'évaluation finale (section 15).

In [ ]:
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Échantillon d'optimisation -- un budget Optuna raisonnable (N_TRIALS_OPTUNA x N_FOLDS_DEFAUT
# fits par modèle) sur les 3M+ lignes complètes serait disproportionné ; un échantillon stratifié
# suffisamment grand donne un signal fiable pour le choix des hyperparamètres, qui sont ensuite
# réutilisés tels quels lors de l'entraînement final sur le train complet (section 14).
X_opt = X_train_ech
y_opt = y_train_ech
cv_optuna = StratifiedKFold(n_splits=N_FOLDS_DEFAUT, shuffle=True, random_state=RANDOM_STATE)


def optimiser(nom_modele: str, fonction_construction, n_trials: int = N_TRIALS_OPTUNA) -> optuna.Study:
    '''Lance une étude Optuna qui maximise le F1 (classe 1) en validation croisée.
    `fonction_construction(trial)` doit renvoyer un pipeline imblearn prêt à être scoré.'''
    def objectif(trial):
        pipe = fonction_construction(trial)
        scores = cross_val_score(pipe, X_opt, y_opt, cv=cv_optuna, scoring="f1", n_jobs=1)
        return scores.mean()

    etude = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE),
        study_name=nom_modele,
    )
    etude.optimize(objectif, n_trials=n_trials, show_progress_bar=False)
    print(f"{nom_modele:20s} meilleur F1(classe 1) CV = {etude.best_value:.4f}  "
          f"params = {etude.best_params}")
    return etude

### Espaces de recherche par famille de modèle

Chaque espace reste raisonnable (5-7 hyperparamètres pertinents) plutôt qu'exhaustif — un espace
trop large dilue le budget d'essais sans gain proportionnel.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier,
)
from imblearn.ensemble import BalancedRandomForestClassifier, EasyEnsembleClassifier
from catboost import CatBoostClassifier

POIDS_CLASSE_1 = (y_train == 0).sum() / max((y_train == 1).sum(), 1)  # ratio pour scale_pos_weight

def espace_logreg(trial):
    clf = LogisticRegression(
        C=trial.suggest_float("C", 1e-3, 10.0, log=True),
        penalty="elasticnet", solver="saga",
        l1_ratio=trial.suggest_float("l1_ratio", 0.0, 1.0),
        class_weight="balanced", max_iter=300, random_state=RANDOM_STATE,
    )
    return construire_pipeline(clf)

def espace_decision_tree(trial):
    clf = DecisionTreeClassifier(
        max_depth=trial.suggest_int("max_depth", 3, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 200, log=True),
        class_weight="balanced", random_state=RANDOM_STATE,
    )
    return construire_pipeline(clf)

def espace_random_forest(trial):
    clf = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50),
        max_depth=trial.suggest_int("max_depth", 4, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 50, log=True),
        max_features=trial.suggest_float("max_features", 0.3, 1.0),
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    )
    return construire_pipeline(clf)

def espace_extra_trees(trial):
    clf = ExtraTreesClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50),
        max_depth=trial.suggest_int("max_depth", 4, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 50, log=True),
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    )
    return construire_pipeline(clf)

def espace_balanced_rf(trial):
    clf = BalancedRandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 400, step=50),
        max_depth=trial.suggest_int("max_depth", 4, 20),
        sampling_strategy="all", replacement=True, bootstrap=False,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    return construire_pipeline(clf, sampler=None)  # rééquilibrage déjà interne au modèle

def espace_easy_ensemble(trial):
    clf = EasyEnsembleClassifier(
        n_estimators=trial.suggest_int("n_estimators", 10, 50, step=10),
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    return construire_pipeline(clf, sampler=None, avec_selection=False)  # ensembles internes -> pas de SelectFromModel

def espace_adaboost(trial):
    clf = AdaBoostClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 300, step=25),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
        random_state=RANDOM_STATE,
    )
    return construire_pipeline(clf)

def espace_gradient_boosting(trial):
    clf = GradientBoostingClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 400, step=50),
        max_depth=trial.suggest_int("max_depth", 2, 8),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        random_state=RANDOM_STATE,
    )
    return construire_pipeline(clf)

def espace_histgb(trial):
    clf = HistGradientBoostingClassifier(
        max_iter=trial.suggest_int("max_iter", 100, 400, step=50),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        l2_regularization=trial.suggest_float("l2_regularization", 0.0, 1.0),
        class_weight="balanced", random_state=RANDOM_STATE,
    )
    return construire_pipeline(clf)

def espace_xgboost(trial):
    clf = XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 150, 600, step=50),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        scale_pos_weight=POIDS_CLASSE_1, eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    return construire_pipeline(clf, sampler=None)

def espace_lightgbm(trial):
    clf = LGBMClassifier(
        n_estimators=trial.suggest_int("n_estimators", 150, 600, step=50),
        max_depth=trial.suggest_int("max_depth", -1, 15),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 127, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
    )
    return construire_pipeline(clf, sampler=None)

def espace_catboost(trial):
    clf = CatBoostClassifier(
        iterations=trial.suggest_int("iterations", 150, 600, step=50),
        depth=trial.suggest_int("depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        auto_class_weights="Balanced", random_state=RANDOM_STATE, verbose=False,
    )
    return construire_pipeline(clf, sampler=None)

ESPACES_RECHERCHE = {
    "LogisticRegression": espace_logreg,
    "DecisionTree": espace_decision_tree,
    "RandomForest": espace_random_forest,
    "ExtraTrees": espace_extra_trees,
    "BalancedRandomForest": espace_balanced_rf,
    "EasyEnsemble": espace_easy_ensemble,
    "AdaBoost": espace_adaboost,
    "GradientBoosting": espace_gradient_boosting,
    "HistGradientBoosting": espace_histgb,
    "XGBoost": espace_xgboost,
    "LightGBM": espace_lightgbm,
    "CatBoost": espace_catboost,
}
print(f"{len(ESPACES_RECHERCHE)} espaces de recherche définis.")

In [ ]:
etudes = {}
for nom_modele, fn_espace in ESPACES_RECHERCHE.items():
    etudes[nom_modele] = optimiser(nom_modele, fn_espace)

meilleurs_params = {nom: etude.best_params for nom, etude in etudes.items()}
with open(REPORTS_DIR / "meilleurs_hyperparametres.json", "w") as f:
    json.dump(meilleurs_params, f, indent=2, default=str)
print(f"\nMeilleurs hyperparamètres sauvegardés : {REPORTS_DIR / 'meilleurs_hyperparametres.json'}")

## 14. Entraînement des modèles

Chaque modèle est réinstancié avec ses meilleurs hyperparamètres (section 13), puis entraîné dans
le **même pipeline** (section 12) sur `X_train`/`y_train` complet (pas l'échantillon réduit utilisé
pour l'optimisation — celui-ci ne servait qu'à choisir les hyperparamètres à budget de calcul
raisonnable). `StratifiedKFold` à 5 **et** 10 folds sont comparés pour vérifier que le classement
des modèles est stable (peu sensible au découpage) avant de figer la comparaison finale.

In [ ]:
def reinstancier_avec_params(nom_modele: str, params: dict):
    '''Reconstruit un pipeline identique à celui de l'espace de recherche Optuna, mais avec les
    meilleurs paramètres trouvés, prêt pour un fit sur le train complet.'''
    if nom_modele == "LogisticRegression":
        clf = LogisticRegression(**params, penalty="elasticnet", solver="saga",
                                  class_weight="balanced", max_iter=300, random_state=RANDOM_STATE)
        return construire_pipeline(clf)
    if nom_modele == "DecisionTree":
        clf = DecisionTreeClassifier(**params, class_weight="balanced", random_state=RANDOM_STATE)
        return construire_pipeline(clf)
    if nom_modele == "RandomForest":
        clf = RandomForestClassifier(**params, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
        return construire_pipeline(clf)
    if nom_modele == "ExtraTrees":
        clf = ExtraTreesClassifier(**params, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
        return construire_pipeline(clf)
    if nom_modele == "BalancedRandomForest":
        clf = BalancedRandomForestClassifier(**params, sampling_strategy="all", replacement=True,
                                              bootstrap=False, random_state=RANDOM_STATE, n_jobs=-1)
        return construire_pipeline(clf, sampler=None)
    if nom_modele == "EasyEnsemble":
        clf = EasyEnsembleClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
        return construire_pipeline(clf, sampler=None, avec_selection=False)
    if nom_modele == "AdaBoost":
        clf = AdaBoostClassifier(**params, random_state=RANDOM_STATE)
        return construire_pipeline(clf)
    if nom_modele == "GradientBoosting":
        clf = GradientBoostingClassifier(**params, random_state=RANDOM_STATE)
        return construire_pipeline(clf)
    if nom_modele == "HistGradientBoosting":
        clf = HistGradientBoostingClassifier(**params, class_weight="balanced", random_state=RANDOM_STATE)
        return construire_pipeline(clf)
    if nom_modele == "XGBoost":
        clf = XGBClassifier(**params, scale_pos_weight=POIDS_CLASSE_1, eval_metric="logloss",
                             random_state=RANDOM_STATE, n_jobs=-1)
        return construire_pipeline(clf, sampler=None)
    if nom_modele == "LightGBM":
        clf = LGBMClassifier(**params, class_weight="balanced", random_state=RANDOM_STATE,
                              n_jobs=-1, verbosity=-1)
        return construire_pipeline(clf, sampler=None)
    if nom_modele == "CatBoost":
        clf = CatBoostClassifier(**params, auto_class_weights="Balanced", random_state=RANDOM_STATE,
                                  verbose=False)
        return construire_pipeline(clf, sampler=None)
    raise ValueError(f"Modèle inconnu : {nom_modele}")

In [ ]:
from sklearn.model_selection import cross_validate

resultats_cv_par_nb_folds = {}
scoring_multiple = {
    "f1": "f1", "recall": "recall", "precision": "precision",
    "balanced_accuracy": "balanced_accuracy", "roc_auc": "roc_auc", "average_precision": "average_precision",
}

for n_folds in N_FOLDS_CANDIDATS:
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    lignes = []
    for nom_modele, params in meilleurs_params.items():
        pipe = reinstancier_avec_params(nom_modele, params)
        cv_res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring_multiple, n_jobs=1)
        lignes.append({
            "algo": nom_modele, "n_folds": n_folds,
            **{m: cv_res[f"test_{m}"].mean() for m in scoring_multiple},
            **{f"{m}_std": cv_res[f"test_{m}"].std() for m in scoring_multiple},
        })
        print(f"[{n_folds} folds] {nom_modele:22s} F1={lignes[-1]['f1']:.4f}  "
              f"PR-AUC={lignes[-1]['average_precision']:.4f}")
    resultats_cv_par_nb_folds[n_folds] = pd.DataFrame(lignes)

stabilite_folds = resultats_cv_par_nb_folds[N_FOLDS_CANDIDATS[0]].set_index("algo")["f1"].corr(
    resultats_cv_par_nb_folds[N_FOLDS_CANDIDATS[-1]].set_index("algo")["f1"]
)
print(f"\nCorrélation du classement F1 entre {N_FOLDS_CANDIDATS[0]} et {N_FOLDS_CANDIDATS[-1]} folds : "
      f"{stabilite_folds:.3f} (proche de 1 = classement stable, quel que soit le découpage)")

resultats_cv = resultats_cv_par_nb_folds[N_FOLDS_DEFAUT]

## 15. Comparaison des modèles

Classement final sur le F1(classe 1) moyen en validation croisée, complété par toutes les
métriques demandées (accuracy, precision, recall, F1, balanced accuracy, MCC, ROC-AUC, PR-AUC),
puis visualisations : matrice de confusion, courbes ROC et Precision-Recall, courbe
d'apprentissage et courbe de calibration pour le modèle en tête.

In [ ]:
from sklearn.metrics import matthews_corrcoef, make_scorer

scorer_mcc = make_scorer(matthews_corrcoef)
mcc_par_modele = {}
for nom_modele, params in meilleurs_params.items():
    pipe = reinstancier_avec_params(nom_modele, params)
    cv = StratifiedKFold(n_splits=N_FOLDS_DEFAUT, shuffle=True, random_state=RANDOM_STATE)
    scores_mcc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=scorer_mcc, n_jobs=1)
    mcc_par_modele[nom_modele] = scores_mcc.mean()

resultats_cv["mcc"] = resultats_cv["algo"].map(mcc_par_modele)
tableau_comparaison = resultats_cv.sort_values("f1", ascending=False).reset_index(drop=True)
tableau_comparaison[["algo", "f1", "average_precision", "recall", "precision",
                      "balanced_accuracy", "mcc", "roc_auc"]]

In [ ]:
NOM_MEILLEUR_MODELE = tableau_comparaison.iloc[0]["algo"]
print(f"Modèle retenu (F1 classe 1 le plus élevé en CV) : {NOM_MEILLEUR_MODELE}")

fig, ax = plt.subplots(figsize=(9, 0.4 * len(tableau_comparaison) + 1))
sns.barplot(data=tableau_comparaison, x="f1", y="algo", ax=ax, color="#2c3e50")
ax.set_title("F1-score (classe 1) par modèle -- validation croisée")
ax.set_xlabel("F1-score (classe 1)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "07_comparaison_modeles_f1.png", dpi=120)
plt.show()

In [ ]:
# Entraînement final de chaque modèle sur X_train complet, puis évaluation honnête sur X_val --
# nécessaire pour tracer les courbes ROC/PR/calibration/confusion (la validation croisée seule
# ne donne que des moyennes, pas de prédictions individuelles réutilisables).
modeles_entraines = {}
for nom_modele, params in meilleurs_params.items():
    pipe = reinstancier_avec_params(nom_modele, params)
    pipe.fit(X_train, y_train)
    modeles_entraines[nom_modele] = pipe
    print(f"{nom_modele:22s} entraîné sur {len(X_train):,} lignes.".replace(",", " "))

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for nom_modele, pipe in modeles_entraines.items():
    probas_val = pipe.predict_proba(X_val)[:, 1]

    fpr, tpr, _ = roc_curve(y_val, probas_val)
    axes[0].plot(fpr, tpr, label=f"{nom_modele} (AUC={auc(fpr, tpr):.3f})")

    precision, recall, _ = precision_recall_curve(y_val, probas_val)
    axes[1].plot(recall, precision, label=f"{nom_modele} (AP={average_precision_score(y_val, probas_val):.3f})")

axes[0].plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
axes[0].set_xlabel("Taux de faux positifs"); axes[0].set_ylabel("Taux de vrais positifs")
axes[0].set_title("Courbes ROC (validation)"); axes[0].legend(fontsize=8)

taux_positifs = y_val.mean()
axes[1].axhline(taux_positifs, linestyle="--", color="grey", linewidth=1, label=f"Hasard ({taux_positifs:.3f})")
axes[1].set_xlabel("Rappel"); axes[1].set_ylabel("Précision")
axes[1].set_title("Courbes Precision-Recall (validation)"); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "08_roc_pr_curves.png", dpi=120)
plt.show()

In [ ]:
meilleur_pipe = modeles_entraines[NOM_MEILLEUR_MODELE]
preds_val_05 = meilleur_pipe.predict(X_val)

print(f"=== {NOM_MEILLEUR_MODELE} -- rapport de classification (seuil 0.5, validation) ===")
print(classification_report(y_val, preds_val_05, target_names=["Non éligible (0)", "Éligible (1)"]))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_val, preds_val_05), annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Prédit 0", "Prédit 1"], yticklabels=["Réel 0", "Réel 1"])
ax.set_title(f"Matrice de confusion -- {NOM_MEILLEUR_MODELE} (seuil 0.5)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "09_confusion_matrix.png", dpi=120)
plt.show()

In [ ]:
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.calibration import CalibrationDisplay

# --- Courbe d'apprentissage (le modèle bénéficierait-il de plus de données ?) -----
tailles, scores_train, scores_val = learning_curve(
    meilleur_pipe, X_train, y_train, cv=3, scoring="f1", n_jobs=1,
    train_sizes=np.linspace(0.1, 1.0, 5), random_state=RANDOM_STATE,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(tailles, scores_train.mean(axis=1), "o-", label="Train")
axes[0].plot(tailles, scores_val.mean(axis=1), "o-", label="Validation (CV)")
axes[0].set_xlabel("Taille du train"); axes[0].set_ylabel("F1 (classe 1)")
axes[0].set_title(f"Courbe d'apprentissage -- {NOM_MEILLEUR_MODELE}")
axes[0].legend()

# --- Courbe de calibration (les probabilités prédites sont-elles fiables ?) -------
CalibrationDisplay.from_predictions(y_val, meilleur_pipe.predict_proba(X_val)[:, 1],
                                     n_bins=15, ax=axes[1])
axes[1].set_title(f"Courbe de calibration -- {NOM_MEILLEUR_MODELE}")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "10_learning_calibration_curves.png", dpi=120)
plt.show()

**Lecture** : si la courbe d'apprentissage montre un score de validation encore en hausse au
maximum de taille de train testée, plus de données améliorerait probablement le modèle — argument
concret pour prioriser l'extension du dataset côté pipeline Spark plutôt que l'ingénierie de
features supplémentaires. La courbe de calibration indique si `predict_proba` peut être interprété
comme une vraie probabilité (utile si le score est utilisé pour prioriser des contacts commerciaux,
pas seulement pour une décision binaire) — un modèle mal calibré resterait exploitable pour le
classement (ranking) mais pas pour lire "70% de chances d'éligibilité" au sens strict.

## 16. Optimisation du seuil de décision

Le seuil 0.5 n'a aucune justification particulière sur une cible aussi déséquilibrée (cf. constat
déjà établi sur le V1.5 : au seuil par défaut, le F1 de la classe positive était quasi nul). Tous
les seuils entre 0.20 et 0.90 sont testés sur **la validation** (jamais le test) ; celui qui
maximise le F1(classe 1) est retenu et **sauvegardé avec le modèle** (section 19) — sans lui, le
modèle rechargé en production retomberait sur `.predict()` à 0.5.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

probas_val_meilleur = meilleur_pipe.predict_proba(X_val)[:, 1]

resultats_seuils = []
for seuil in SEUILS_A_TESTER:
    preds = (probas_val_meilleur >= seuil).astype(int)
    resultats_seuils.append({
        "seuil": seuil,
        "f1": f1_score(y_val, preds),
        "precision": precision_score(y_val, preds, zero_division=0),
        "recall": recall_score(y_val, preds, zero_division=0),
    })
resultats_seuils = pd.DataFrame(resultats_seuils)

SEUIL_OPTIMAL = float(resultats_seuils.loc[resultats_seuils["f1"].idxmax(), "seuil"])
f1_au_seuil_optimal = resultats_seuils["f1"].max()
print(f"Seuil optimal (F1 classe 1 max) : {SEUIL_OPTIMAL:.2f}  (F1={f1_au_seuil_optimal:.4f})")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(resultats_seuils["seuil"], resultats_seuils["f1"], label="F1 (classe 1)", linewidth=2)
ax.plot(resultats_seuils["seuil"], resultats_seuils["precision"], label="Précision", linestyle="--")
ax.plot(resultats_seuils["seuil"], resultats_seuils["recall"], label="Rappel", linestyle="--")
ax.axvline(SEUIL_OPTIMAL, color="red", linestyle=":", label=f"Seuil retenu = {SEUIL_OPTIMAL:.2f}")
ax.axvline(0.5, color="grey", linestyle=":", alpha=0.6, label="Seuil par défaut (0.5)")
ax.set_xlabel("Seuil de décision"); ax.set_ylabel("Score")
ax.set_title(f"Optimisation du seuil -- {NOM_MEILLEUR_MODELE}")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "11_optimisation_seuil.png", dpi=120)
plt.show()

**Note pour la Direction métier** : le seuil ci-dessus optimise un compromis précision/rappel
*symétrique* (F1). Si une matrice de coût métier est communiquée (coût d'un contact commercial
manqué vs. coût d'un contact inutile), le seuil peut être redéplacé vers la courbe de précision ou
de rappel ci-dessus sans réentraîner le modèle — c'est tout l'intérêt de séparer seuil et
entraînement.

## 17. Interprétabilité

Comprendre **pourquoi** le modèle retenu (`NOM_MEILLEUR_MODELE`) prend ses décisions, avant toute
mise en production : importance globale (SHAP, permutation), et effet marginal des variables les
plus influentes (PDP) — utile pour détecter un signal qui n'aurait pas de sens métier (indice
possible de fuite de données, cf. avertissement section 5.9 sur `CUSTOMER_RATING`).

In [ ]:
import shap

# SHAP nécessite d'accéder au modèle final ET aux données déjà transformées par le préprocesseur
# du pipeline -- on extrait les deux, sans jamais refitter (le pipeline est déjà entraîné, section 15).
preprocesseur_fit = meilleur_pipe.named_steps["prep"]
etape_select = meilleur_pipe.named_steps.get("select")
modele_final = meilleur_pipe.named_steps["clf"]

# Échantillon de validation pour SHAP (coût de calcul TreeExplainer raisonnable jusqu'à
# quelques milliers de lignes -- pas besoin du X_val complet pour un diagnostic fiable).
X_val_ech = X_val.sample(n=min(3_000, len(X_val)), random_state=RANDOM_STATE)
X_val_transforme = preprocesseur_fit.transform(X_val_ech)
noms_apres_prep = preprocesseur_fit.get_feature_names_out()
if hasattr(X_val_transforme, "toarray"):
    X_val_transforme = X_val_transforme.toarray()
X_val_transforme = pd.DataFrame(X_val_transforme, columns=noms_apres_prep)

if etape_select is not None:
    masque_select = etape_select.get_support()
    X_val_transforme = X_val_transforme.loc[:, masque_select]

explainer = shap.TreeExplainer(modele_final) if hasattr(modele_final, "get_booster") or hasattr(modele_final, "booster_") \
    else shap.Explainer(modele_final, X_val_transforme)
valeurs_shap = explainer(X_val_transforme)

shap.summary_plot(valeurs_shap, X_val_transforme, show=False, max_display=20)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "12_shap_summary.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Importance SHAP moyenne (|valeur|) -- vue synthétique, complémentaire au summary plot ci-dessus.
importance_shap = pd.Series(
    np.abs(valeurs_shap.values).mean(axis=0), index=X_val_transforme.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, max(4, 0.3 * min(20, len(importance_shap)))))
sns.barplot(x=importance_shap.values[:20], y=importance_shap.index[:20], ax=ax, color="#16a085")
ax.set_title(f"Top 20 -- importance SHAP moyenne ({NOM_MEILLEUR_MODELE})")
plt.tight_layout()
plt.show()

importance_shap.head(20).to_frame("importance_shap_moyenne")

In [ ]:
# Permutation importance -- confirme (ou nuance) SHAP avec une méthode indépendante du modèle,
# directement sur le score F1 (plus proche de la métrique métier que la valeur SHAP brute).
perm_finale = permutation_importance(
    meilleur_pipe, X_val_ech, y_val.loc[X_val_ech.index],
    scoring="f1", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
importance_permutation_finale = pd.Series(
    perm_finale.importances_mean, index=X_val_ech.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, max(4, 0.3 * min(20, len(importance_permutation_finale)))))
sns.barplot(x=importance_permutation_finale.values[:20], y=importance_permutation_finale.index[:20],
            ax=ax, color="#e67e22")
ax.set_title(f"Top 20 -- Permutation Importance ({NOM_MEILLEUR_MODELE}, scoring=F1)")
plt.tight_layout()
plt.show()

In [ ]:
# Partial Dependence Plots -- effet marginal des 4 variables les plus importantes (SHAP).
from sklearn.inspection import PartialDependenceDisplay

top4_features = [c for c in importance_shap.index[:4] if c in X_val_transforme.columns]
if top4_features:
    fig, ax = plt.subplots(figsize=(14, 3.2))
    PartialDependenceDisplay.from_estimator(
        modele_final, X_val_transforme, top4_features, ax=ax, n_jobs=-1
    )
    plt.suptitle(f"Partial Dependence Plots -- {NOM_MEILLEUR_MODELE}", y=1.05)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "13_pdp.png", dpi=120)
    plt.show()

**Lecture métier** : les variables en tête (SHAP + permutation, si elles convergent) doivent
correspondre à une intuition métier plausible (ex. `pack_actuel`/`pack_etat` reflétant
l'engagement client, `CUSTOMER_RATING` reflétant la solvabilité). Si une variable technique ou
sans justification métier apparaît en tête (ex. un identifiant mal filtré), c'est un signal fort à
vérifier avant toute mise en production — plus fiable qu'une simple performance élevée pour
détecter une fuite de données, cf. avertissement section 5.9.

## 18. Analyse des erreurs

Regarder concrètement les faux positifs et faux négatifs (au seuil optimal, section 16) pour
identifier des motifs récurrents — utile pour prioriser un futur feature engineering ciblé plutôt
que d'itérer à l'aveugle.

In [ ]:
preds_val_seuil_optimal = (probas_val_meilleur >= SEUIL_OPTIMAL).astype(int)

analyse_erreurs = X_val.copy()
analyse_erreurs["reel"] = y_val.values
analyse_erreurs["predit"] = preds_val_seuil_optimal
analyse_erreurs["probabilite"] = probas_val_meilleur

faux_positifs = analyse_erreurs[(analyse_erreurs["reel"] == 0) & (analyse_erreurs["predit"] == 1)]
faux_negatifs = analyse_erreurs[(analyse_erreurs["reel"] == 1) & (analyse_erreurs["predit"] == 0)]
vrais_positifs = analyse_erreurs[(analyse_erreurs["reel"] == 1) & (analyse_erreurs["predit"] == 1)]

print(f"Faux positifs : {len(faux_positifs):,} ({len(faux_positifs) / len(analyse_erreurs):.2%} du set)".replace(",", " "))
print(f"Faux négatifs : {len(faux_negatifs):,} ({len(faux_negatifs) / len(analyse_erreurs):.2%} du set)".replace(",", " "))
print(f"Vrais positifs: {len(vrais_positifs):,}".replace(",", " "))

In [ ]:
# Comparaison des distributions des variables les plus importantes : vrais positifs vs faux
# positifs vs faux négatifs -- fait ressortir ce qui distingue une erreur d'une bonne prédiction.
top_features_erreurs = [c for c in importance_shap.index[:6] if c in analyse_erreurs.columns][:4]
# Repli sur les features numériques les plus corrélées à la cible si aucune ne correspond
# directement (cas où SHAP porte sur des colonnes déjà transformées, ex. one-hot).
if not top_features_erreurs:
    top_features_erreurs = correlations_cible.index[:4].tolist()

if top_features_erreurs:
    fig, axes = plt.subplots(1, len(top_features_erreurs), figsize=(5 * len(top_features_erreurs), 4))
    axes = np.atleast_1d(axes)
    groupes = {"Vrai positif": vrais_positifs, "Faux positif": faux_positifs, "Faux négatif": faux_negatifs}
    for ax, col in zip(axes, top_features_erreurs):
        for nom_groupe, sous_df in groupes.items():
            if col in sous_df.columns and sous_df[col].notna().any():
                sns.kdeplot(sous_df[col].dropna(), ax=ax, label=nom_groupe)
        ax.set_title(col, fontsize=10)
        ax.legend(fontsize=7)
    plt.suptitle("Distribution des variables clés selon le type de prédiction", y=1.03)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "14_analyse_erreurs.png", dpi=120)
    plt.show()

In [ ]:
print("Faux positifs les plus confiants (le modèle s'est trompé avec une forte probabilité) :")
faux_positifs.sort_values("probabilite", ascending=False).head(10)

In [ ]:
print("Faux négatifs les plus confiants dans l'autre sens (probabilité la plus basse malgré la cible=1) :")
faux_negatifs.sort_values("probabilite", ascending=True).head(10)

**Pistes d'amélioration identifiées** (à confirmer sur un volume d'erreurs plus large que
cet aperçu) :
- Si les faux négatifs se concentrent sur une plage étroite de `CUSTOMER_RATING`/`pack_actuel`, une
  variable d'interaction dédiée à cette zone (section 8) pourrait aider à mieux les séparer.
- Si les faux positifs sont concentrés sur une modalité de `BPR` (agence) ou `CODE_VILLE`
  particulière, cela peut indiquer un effet régional non capturé par les features actuelles — ou,
  à l'inverse, une réelle appétence locale que le label historique ne reflète pas encore
  (temporalité du dataset à vérifier avec le métier).
- Les faux positifs les plus confiants sont les clients à recontacter en priorité même s'ils sont
  classés "non éligibles" par la vérité terrain historique — souvent les cas limites les plus
  intéressants commercialement.

## 19. Sauvegarde du meilleur modèle

Avant la sauvegarde définitive, le pipeline retenu est **refit sur `X_train_full`**
(train + validation réunis, soit 85 % du dataset) — la validation n'a servi qu'au tuning
(section 13) et au choix du seuil (section 16), elle peut donc être réintégrée à l'entraînement
final sans invalider l'évaluation, qui reste basée sur `X_test`, jamais vu jusqu'ici (section 15).
Le `Pipeline` complet, le seuil optimal et les métadonnées de traçabilité sont sauvegardés
ensemble via `joblib` (recommandé pour les objets scikit-learn/imblearn volumineux) — un export
`pickle` brut est fourni en complément pour compatibilité avec des environnements sans `joblib`.

In [ ]:
import pickle
from datetime import datetime, timezone

# --- Évaluation finale honnête sur le test, jamais vu jusqu'ici -------------
meilleur_pipe_final = reinstancier_avec_params(NOM_MEILLEUR_MODELE, meilleurs_params[NOM_MEILLEUR_MODELE])
meilleur_pipe_final.fit(X_train_full, y_train_full)

probas_test = meilleur_pipe_final.predict_proba(X_test)[:, 1]
preds_test_seuil_optimal = (probas_test >= SEUIL_OPTIMAL).astype(int)

print(f"=== Évaluation finale sur le TEST (jamais vu) -- {NOM_MEILLEUR_MODELE}, seuil={SEUIL_OPTIMAL:.2f} ===")
print(classification_report(y_test, preds_test_seuil_optimal, target_names=["Non éligible (0)", "Éligible (1)"]))
print(f"PR-AUC (test)  : {average_precision_score(y_test, probas_test):.4f}")
print(f"ROC-AUC (test) : {auc(*roc_curve(y_test, probas_test)[:2]):.4f}")
print(f"MCC (test)     : {matthews_corrcoef(y_test, preds_test_seuil_optimal):.4f}")

In [ ]:
VERSION_MODELE = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
CHEMIN_MODELE = MODELS_DIR / f"modele_eligibilite_{NOM_MEILLEUR_MODELE}_{VERSION_MODELE}"

metadonnees = {
    "nom_modele": NOM_MEILLEUR_MODELE,
    "version": VERSION_MODELE,
    "seuil_decision": SEUIL_OPTIMAL,
    "hyperparametres": meilleurs_params[NOM_MEILLEUR_MODELE],
    "colonnes_features_entree": list(X_train_full.columns),
    "metriques_test": {
        "f1_classe1": float(f1_score(y_test, preds_test_seuil_optimal)),
        "precision_classe1": float(precision_score(y_test, preds_test_seuil_optimal, zero_division=0)),
        "recall_classe1": float(recall_score(y_test, preds_test_seuil_optimal, zero_division=0)),
        "pr_auc": float(average_precision_score(y_test, probas_test)),
        "roc_auc": float(auc(*roc_curve(y_test, probas_test)[:2])),
        "mcc": float(matthews_corrcoef(y_test, preds_test_seuil_optimal)),
    },
    "random_state": RANDOM_STATE,
    "taille_train_final": len(X_train_full),
    "taux_positifs_train": float(y_train_full.mean()),
}

# joblib (recommandé -- compression + gestion efficace des tableaux numpy volumineux)
import joblib
joblib.dump(meilleur_pipe_final, f"{CHEMIN_MODELE}.joblib", compress=3)
joblib.dump(metadonnees, f"{CHEMIN_MODELE}_meta.joblib")

# pickle (complément -- compatibilité maximale)
with open(f"{CHEMIN_MODELE}.pkl", "wb") as f:
    pickle.dump(meilleur_pipe_final, f)
with open(f"{CHEMIN_MODELE}_meta.json", "w") as f:
    json.dump(metadonnees, f, indent=2, default=str)

print(f"Pipeline complet sauvegardé  : {CHEMIN_MODELE}.joblib / .pkl")
print(f"Métadonnées sauvegardées     : {CHEMIN_MODELE}_meta.joblib / _meta.json")
print(f"\nContenu des métadonnées :")
print(json.dumps(metadonnees, indent=2, default=str))

**Rechargement en production (référence)** :

```python
import joblib
pipeline = joblib.load("modele_eligibilite_XGBoost_20260731_120000.joblib")
meta = joblib.load("modele_eligibilite_XGBoost_20260731_120000_meta.joblib")

probas = pipeline.predict_proba(nouveau_dataframe)[:, 1]
predictions = (probas >= meta["seuil_decision"]).astype(int)
```

Aucun `.fit()` ici — uniquement le chargement d'objets déjà appris sur le train, exactement comme
pour le `PipelineModel` MLlib côté Spark (même principe de non-fuite, cf. section Introduction).

## 20. Conclusion

### Ce que ce notebook a établi

- Une EDA complète documentant la qualité, la distribution et le sens métier des variables du
  dataset d'éligibilité.
- Une comparaison **empirique**, pas arbitraire, des choix de préparation (encodage, imputation,
  rééquilibrage, sélection de variables) — chaque décision repose sur un score, pas une intuition.
- Un comparatif de 12 familles de modèles, chacune optimisée par recherche bayésienne (Optuna) sur
  le F1 de la classe positive, la métrique alignée avec l'objectif métier (section 2).
- Un seuil de décision optimisé plutôt que la valeur par défaut 0.5, dont on sait déjà (V1.5)
  qu'elle était inadaptée à ce niveau de déséquilibre.
- Une lecture d'interprétabilité (SHAP, permutation, PDP) et une analyse d'erreurs qui donnent des
  pistes concrètes d'amélioration, pas seulement un score final.

### Limites et prochaines étapes

- **Volumétrie** : plusieurs comparatifs (sections 8, 10, 11, 13) s'appuient sur un échantillon
  stratifié pour rester exécutables en un temps raisonnable — à revalider sur le dataset complet
  (3M+ lignes) avant la mise en production finale, en particulier pour les méthodes de
  rééquilibrage coûteuses (`SVMSMOTE`, `KMeansSMOTE`, `ClusterCentroids`) écartées du run complet.
- **`CUSTOMER_RATING`** (section 5.9) : le risque de fuite de données doit être confirmé/infirmé
  avec l'équipe métier avant mise en production — actuellement conservé mais documenté.
- **Matrice de coût métier** : le seuil actuel optimise un F1 symétrique ; un chiffrage du coût
  d'un faux positif vs. faux négatif permettrait un seuil réellement optimal pour la banque.
- **Alignement avec le pipeline Spark/MLlib** : ce notebook ne remplace pas l'encodage de
  production (StringIndexer/OneHotEncoder Spark, cf. `pipeline_training_v1_5.ipynb`) — il sert de
  **référence de performance atteignable** en pur Python/sklearn. Si le modèle retenu ici
  (`NOM_MEILLEUR_MODELE`) dépasse significativement le RandomForest MLlib actuel, la discussion à
  ouvrir avec l'équipe infrastructure est : réentraîner ce modèle sklearn/boosting sur l'intégralité
  du dataset (nécessite de sortir du tout-Spark, ou d'utiliser XGBoost4J-Spark/SynapseML pour rester
  distribué), ou accepter un léger écart de performance en échange de la scalabilité MLlib native.
- **Monitoring en production** : à mettre en place (dérive de distribution des features, dérive de
  performance) une fois le modèle déployé — hors périmètre de ce notebook d'entraînement.